# Hypotension obs24_target8_gap0 Cluster Visualization

Load saved autoencoder embeddings, cluster assignments, importance maps, and prepared tensors for the `vit_baseline/obs24_target8_gap0/hypotension` run, then render cluster heatmaps with normal-range coloring.

## Imports and Paths

In [ ]:
from __future__ import annotations

import json
import logging
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "interpretable_ts_vit").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from interpretable_ts_vit.binning import TimeSeriesBinner
from interpretable_ts_vit.io import load_split
from interpretable_ts_vit.visualization import (
    aggregate_cluster_value_matrices,
    cluster_assignment_counts,
    patient_value_matrix,
    plot_value_heatmap,
    value_ranges_by_variable,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
)
logger = logging.getLogger("hypotension_cluster_visualization")
logger.info("Project root: %s", PROJECT_ROOT)

In [ ]:
EXPERIMENT_NAME = "vit_baseline"
TARGET = "hypotension"
WINDOW = "obs24_target8_gap0"
SPLIT = "test"

RUN_DIR = PROJECT_ROOT / "runs" / "mimic_targets" / EXPERIMENT_NAME / WINDOW / TARGET
PROCESSED_DIR = PROJECT_ROOT / "data" / "mimic_targets" / "processed" / WINDOW / TARGET
CLUSTERS_DIR = RUN_DIR / "clusters" / SPLIT
EXPLANATIONS_DIR = RUN_DIR / "explanations" / SPLIT
OUTPUT_DIR = RUN_DIR / "cluster_heatmaps" / SPLIT
SELECTED_OUTPUT_DIR = RUN_DIR / "selected_cluster_heatmaps" / SPLIT
SELECTED_PATIENT_OUTPUT_DIR = RUN_DIR / "selected_patient_heatmaps" / SPLIT
NORMAL_RANGES_PATH = PROJECT_ROOT / "src" / "interpretable_ts_vit" / "normal_ranges.json"

PLOT_MODE = "value_with_importance_opacity"
IMPORTANCE_THRESHOLD = None
SHOW_VALUES = True
USE_NORMAL_RANGES = True

RUN_DIR

## Load Saved Artifacts

In [ ]:
def require_artifacts(paths):
    missing = [f"{name}: {path}" for name, path in paths.items() if not Path(path).exists()]
    if missing:
        raise FileNotFoundError("Missing required artifact(s):\n" + "\n".join(missing))


required = {
    "prepared binner": PROCESSED_DIR / "binner.json",
    f"prepared {SPLIT} split": PROCESSED_DIR / f"{SPLIT}.npz",
    "run binner": RUN_DIR / "binner.json",
    f"run {SPLIT} split": RUN_DIR / f"{SPLIT}.npz",
    "autoencoder embeddings": CLUSTERS_DIR / "autoencoder_embeddings.csv",
    "cluster assignments": CLUSTERS_DIR / "cluster_assignments.csv",
    "cluster centroids": CLUSTERS_DIR / "cluster_centroids.csv",
    "cluster metadata": CLUSTERS_DIR / "cluster_metadata.json",
    f"{SPLIT} explanations": EXPLANATIONS_DIR,
    "normal ranges": NORMAL_RANGES_PATH,
}
require_artifacts(required)

binner = TimeSeriesBinner.load(PROCESSED_DIR / "binner.json")
dataset = load_split(PROCESSED_DIR / f"{SPLIT}.npz")
embeddings = pd.read_csv(CLUSTERS_DIR / "autoencoder_embeddings.csv", dtype={"patient_id": str})
assignments = pd.read_csv(CLUSTERS_DIR / "cluster_assignments.csv", dtype={"patient_id": str})
centroids = pd.read_csv(CLUSTERS_DIR / "cluster_centroids.csv", dtype={"patient_id": str})
cluster_metadata = json.loads((CLUSTERS_DIR / "cluster_metadata.json").read_text(encoding="utf-8"))

display(pd.DataFrame([
    {"artifact": "embeddings", "path": str(CLUSTERS_DIR / "autoencoder_embeddings.csv"), "rows": len(embeddings)},
    {"artifact": "assignments", "path": str(CLUSTERS_DIR / "cluster_assignments.csv"), "rows": len(assignments)},
    {"artifact": "centroids", "path": str(CLUSTERS_DIR / "cluster_centroids.csv"), "rows": len(centroids)},
    {"artifact": "prepared split", "path": str(PROCESSED_DIR / f"{SPLIT}.npz"), "rows": len(dataset)},
]))
display(assignments.groupby(["predicted_label", "cluster"]).size().rename("patients").reset_index())
display(embeddings.head())
cluster_metadata

## Plot Helpers

In [ ]:
def safe_path_component(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_") or "class"


def importance_style_from_plot_mode(plot_mode):
    if plot_mode == "value_with_importance_opacity":
        return "opacity"
    if plot_mode == "value_with_importance_border":
        return "border"
    if plot_mode == "value":
        return None
    raise ValueError("PLOT_MODE must be 'value', 'value_with_importance_opacity', or 'value_with_importance_border'.")


def cluster_key_from_group_key(group_key, group_columns):
    if group_columns == ["cluster"]:
        return int(group_key)
    predicted_label, cluster = group_key
    return str(predicted_label), int(cluster)


def heatmap_path(output_dir, cluster_key):
    if isinstance(cluster_key, tuple):
        predicted_label, cluster = cluster_key
        return output_dir / safe_path_component(predicted_label) / f"cluster_{cluster}.png"
    return output_dir / f"cluster_{cluster_key}.png"


def importance_matrix_path(cluster_key):
    if isinstance(cluster_key, tuple):
        predicted_label, cluster = cluster_key
        return CLUSTERS_DIR / safe_path_component(predicted_label) / f"cluster_{cluster}.npy"
    return CLUSTERS_DIR / f"cluster_{cluster_key}.npy"


def cluster_title(cluster_key, count):
    suffix = f" (n={count})" if count is not None else ""
    if isinstance(cluster_key, tuple):
        predicted_label, cluster = cluster_key
        return f"Predicted class {predicted_label}: cluster_{cluster}{suffix}"
    return f"cluster_{cluster_key}{suffix}"


def normal_ranges_arg():
    return NORMAL_RANGES_PATH if USE_NORMAL_RANGES else None


def load_importance_matrix(cluster_key):
    path = importance_matrix_path(cluster_key)
    if path.exists():
        return np.load(path)
    logger.warning("Missing importance matrix: %s", path)
    return None


def render_cluster_heatmap(cluster_key, matrix, output_dir, count=None, vmin=None, vmax=None, display_image=True):
    importance_style = importance_style_from_plot_mode(PLOT_MODE)
    importance_matrix = load_importance_matrix(cluster_key) if importance_style is not None else None
    if vmin is None or vmax is None:
        vmin, vmax = value_ranges_by_variable([matrix])
    output_path = heatmap_path(output_dir, cluster_key)
    plot_value_heatmap(
        matrix,
        binner.variable_vocab_,
        binner.time_bins_,
        output_path,
        title=cluster_title(cluster_key, count),
        vmin=vmin,
        vmax=vmax,
        importance_matrix=importance_matrix,
        importance_style=importance_style or "opacity",
        importance_threshold=IMPORTANCE_THRESHOLD,
        show_values=SHOW_VALUES,
        normal_ranges=normal_ranges_arg(),
    )
    if display_image:
        print(output_path)
        display(Image(filename=str(output_path)))
    return output_path


def cluster_value_matrices(assignments_frame=None):
    frame = assignments if assignments_frame is None else assignments_frame
    return aggregate_cluster_value_matrices(dataset, frame, binner)


def plot_all_clusters(display_images=True):
    matrices_by_cluster = cluster_value_matrices()
    counts_by_cluster = cluster_assignment_counts(assignments)
    if not matrices_by_cluster:
        raise ValueError("No cluster value matrices were created.")
    vmin, vmax = value_ranges_by_variable(list(matrices_by_cluster.values()))
    paths = []
    for cluster_key, matrix in matrices_by_cluster.items():
        paths.append(render_cluster_heatmap(
            cluster_key,
            matrix,
            OUTPUT_DIR,
            count=counts_by_cluster.get(cluster_key),
            vmin=vmin,
            vmax=vmax,
            display_image=display_images,
        ))
    return paths


def plot_selected_cluster(class_label, cluster_number):
    class_label = str(class_label)
    cluster_number = int(cluster_number)
    selected = assignments[
        (assignments["predicted_label"].astype(str) == class_label)
        & (assignments["cluster"].astype(int) == cluster_number)
    ].copy()
    if selected.empty:
        available = assignments.groupby(["predicted_label", "cluster"]).size().rename("patients").reset_index()
        display(available)
        raise ValueError(f"No patients found for predicted_label={class_label!r}, cluster={cluster_number}.")
    matrices = cluster_value_matrices(selected)
    cluster_key = (class_label, cluster_number)
    matrix = matrices.get(cluster_key)
    if matrix is None:
        matrix = next(iter(matrices.values()))
    return render_cluster_heatmap(cluster_key, matrix, SELECTED_OUTPUT_DIR, count=len(selected))


def patients_in_cluster(class_label, cluster_number):
    class_label = str(class_label)
    cluster_number = int(cluster_number)
    selected = assignments[
        (assignments["predicted_label"].astype(str) == class_label)
        & (assignments["cluster"].astype(int) == cluster_number)
    ].copy()
    if selected.empty:
        available = assignments.groupby(["predicted_label", "cluster"]).size().rename("patients").reset_index()
        display(available)
        raise ValueError(f"No patients found for predicted_label={class_label!r}, cluster={cluster_number}.")
    return selected.sort_values(["is_centroid", "distance_to_centroid"], ascending=[False, True]).reset_index(drop=True)


def choose_patient_from_cluster(class_label, cluster_number, patient_id=None):
    selected = patients_in_cluster(class_label, cluster_number)
    if patient_id is None:
        centroid_rows = selected[selected["is_centroid"].astype(bool)]
        return str((centroid_rows if not centroid_rows.empty else selected).iloc[0]["patient_id"])
    patient_id = str(patient_id)
    if patient_id not in set(selected["patient_id"].astype(str)):
        display(selected.head(25))
        raise ValueError(f"Patient {patient_id!r} is not in predicted_label={class_label!r}, cluster={cluster_number}.")
    return patient_id


def plot_patient_from_cluster(class_label, cluster_number, patient_id=None):
    class_label = str(class_label)
    cluster_number = int(cluster_number)
    patient_id = choose_patient_from_cluster(class_label, cluster_number, patient_id)
    matrix = patient_value_matrix(dataset, binner, patient_id)
    explanation_path = EXPLANATIONS_DIR / f"{patient_id}.npy"
    importance_matrix = np.load(explanation_path) if explanation_path.exists() else None
    if importance_matrix is None:
        logger.warning("No patient explanation found: %s", explanation_path)
    importance_style = importance_style_from_plot_mode(PLOT_MODE)
    vmin, vmax = value_ranges_by_variable([matrix])
    output_path = SELECTED_PATIENT_OUTPUT_DIR / safe_path_component(class_label) / f"cluster_{cluster_number}" / f"patient_{safe_path_component(patient_id)}.png"
    plot_value_heatmap(
        matrix,
        binner.variable_vocab_,
        binner.time_bins_,
        output_path,
        title=f"Predicted class {class_label}: cluster_{cluster_number}, patient {patient_id}",
        vmin=vmin,
        vmax=vmax,
        importance_matrix=importance_matrix,
        importance_style=importance_style or "opacity",
        importance_threshold=IMPORTANCE_THRESHOLD,
        show_values=SHOW_VALUES,
        normal_ranges=normal_ranges_arg(),
    )
    print(output_path)
    display(Image(filename=str(output_path)))
    return output_path


def plot_cluster_and_patient(class_label, cluster_number, patient_id=None):
    patient_id = choose_patient_from_cluster(class_label, cluster_number, patient_id)
    return {
        "cluster": plot_selected_cluster(class_label, cluster_number),
        "patient": plot_patient_from_cluster(class_label, cluster_number, patient_id),
    }

## Plot All Clusters

In [ ]:
all_cluster_heatmaps = plot_all_clusters(display_images=True)
len(all_cluster_heatmaps)

## Plot One Cluster

In [ ]:
# Choose any available predicted_label / cluster pair from the table above.
# patients_in_cluster("True", 0).head()
# plot_selected_cluster("True", 0)
# plot_patient_from_cluster("True", 0)  # defaults to the centroid patient
# plot_patient_from_cluster("True", 0, patient_id="31225558")
# plot_cluster_and_patient("False", 3)

## Artifact Check

In [ ]:
important_artifacts = [
    CLUSTERS_DIR / "autoencoder_embeddings.csv",
    CLUSTERS_DIR / "cluster_assignments.csv",
    CLUSTERS_DIR / "cluster_centroids.csv",
    CLUSTERS_DIR / "cluster_metadata.json",
    OUTPUT_DIR,
    SELECTED_OUTPUT_DIR,
    SELECTED_PATIENT_OUTPUT_DIR,
]
display(pd.DataFrame({"exists": [path.exists() for path in important_artifacts], "path": [str(path) for path in important_artifacts]}))